## Importing Required Libraries

In [1]:
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torchmetrics

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

## Loading Data

In [2]:
from src.config import DataConfig

data_config = DataConfig()

arabic_dataset = load_dataset(data_config.dataset_name, streaming=True)
train_data = arabic_dataset['train'].take(data_config.train_size)
val_data = arabic_dataset['dev'].take(data_config.val_size)
test_data = arabic_dataset['test'].take(data_config.test_size)

## Inscepcting Data

In [3]:
arabic_dataset

IterableDatasetDict({
    train: IterableDataset({
        features: ['diacratized', 'text'],
        num_shards: 3
    })
    test: IterableDataset({
        features: ['diacratized', 'text'],
        num_shards: 1
    })
    dev: IterableDataset({
        features: ['diacratized', 'text'],
        num_shards: 1
    })
})

In [4]:
first_train_item = next(iter(train_data))
print(first_train_item['text'])
print(first_train_item['diacratized'])

ليس بتضمين ولو اغتصبه إنسان من السارق فهلك في يده بعد القطع فلا ضمان للسارق ولا للمسروق منه أما السارق فلأنه ليس بمالك وأما المالك فلأن العصمة الثابتة له حقا قد بطلت قال القدوري وكان للمولى أن يضمنه الغاصب لأنه لو ضمن لا يرجع بالضمان على السارق وعلى هذا يخرج ما إذا سرق ثوبا فخرقه في الدار خرق
لَيْسَ بِتَضْمِينٍ وَلَوْ اغْتَصَبَهُ إنْسَانٌ مِنْ السَارِقِ فَهَلَكَ فِي يَدِهِ بَعْدَ الْقَطْعِ فَلَا ضَمَانَ لِلسَارِقِ وَلَا لِلْمَسْرُوقِ مِنْهُ أَمَا السَارِقُ فَلِأَنَهُ لَيْسَ بِمَالِكٍ وَأَمَا الْمَالِكُ فَلِأَنَ الْعِصْمَةَ الثَابِتَةَ لَهُ حَقًا قَدْ بَطَلَتْ قَالَ الْقُدُورِيُ وَكَانَ لِلْمَوْلَى أَنْ يُضَمِنَهُ الْغَاصِبَ لِأَنَهُ لَوْ ضَمِنَ لَا يَرْجِعُ بِالضَمَانِ عَلَى السَارِقِ وَعَلَى هَذَا يَخْرُجُ مَا إذَا سَرَقَ ثَوْبًا فَخَرَقَهُ فِي الدَارِ خَرْق


## 

## Prepare data for modeling

In [5]:
from src.tokenizer import ArabTokenizer
from src.config import PathConfig


tokenizer = ArabTokenizer(train_data)
print('vocab size:', tokenizer.vocab_size)
print('num labels:', tokenizer.num_labels)

vocab size: 58
num labels: 10


In [6]:
'../'+PathConfig.tokenizer_path

'../assets/tokenizer.json'

In [7]:
tokenizer.save('../'+PathConfig.tokenizer_path)

In [8]:
from src.dataset import TashkeelDataset

windows_size = DataConfig.window_size
train_dataset = TashkeelDataset(train_data, tokenizer, window_size=windows_size)
val_dataset = TashkeelDataset(val_data, tokenizer, window_size=windows_size)
test_dataset = TashkeelDataset(test_data, tokenizer, window_size=windows_size)

tokenizing row 49000
tokenizing row 9000
tokenizing row 9000


In [9]:
from torch.utils.data import  DataLoader
from src.dataset import collate_fn
from src.config import TrainConfig

train_config = TrainConfig()

train_loader = DataLoader(train_dataset, batch_size=train_config.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=train_config.batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=train_config.batch_size, shuffle=False, collate_fn=collate_fn)

In [10]:
next(iter(train_loader))

(tensor([[32, 44, 45,  ...,  3, 48, 44],
         [47,  3, 20,  ..., 24,  3, 42],
         [44, 22,  3,  ..., 41,  3, 48],
         ...,
         [48,  3, 45,  ..., 44, 47,  3],
         [ 3, 20, 44,  ...,  0,  0,  0],
         [47, 48,  3,  ...,  3, 20, 44]]),
 tensor([[9, 7, 7,  ..., 1, 5, 7],
         [6, 1, 1,  ..., 5, 1, 5],
         [5, 7, 1,  ..., 7, 1, 5],
         ...,
         [5, 1, 7,  ..., 5, 7, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         [6, 5, 1,  ..., 1, 1, 9]]))

## Model Building

In [11]:
from src.model import DiacritizationModelBiGRU
from src.config import ModelConfig

model_config = ModelConfig()

torch.manual_seed(train_config.seed)
model = DiacritizationModelBiGRU.from_config(model_config, vocab_size=tokenizer.vocab_size, labels_size=tokenizer.num_labels)

In [12]:
from src.config import DEVICE
state_dict = torch.load('../'+PathConfig.model_path, weights_only=True, map_location=torch.device(DEVICE))
model.load_state_dict(state_dict)

<All keys matched successfully>

In [13]:
# from src.train import train

# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.NAdam(model.parameters())
# accuracy = torchmetrics.Accuracy(
#     task="multiclass", num_classes=tokenizer.num_labels, ignore_index=0
# ).to(DEVICE)

# train(model, train_loader, val_loader, criterion, optimizer, train_config.n_epochs, accuracy, DEVICE)

## Model Inference

In [14]:
from src.infer import Diacritizer

diacritizer = Diacritizer(model, tokenizer, device=DEVICE)
text = 'كان العرب يحبون العلم ويقدرون العلماء'
dediacritized_text = diacritizer.diacritize(text)
print("Input text:", text)
print("Diacritized text:", dediacritized_text)

62
Input text: كان العرب يحبون العلم ويقدرون العلماء
Diacritized text: كَانَ الْعَرَبُ يُحِبُونَ الْعِلْمَ وَيُقْدِرُونَ الْعُلَمَاءَ


## Model Evaluation

In [15]:
from src.evaluate import evaluate, evaluate_arabic_diacritization

accuracy = torchmetrics.Accuracy(
    task="multiclass", num_classes=tokenizer.num_labels, ignore_index=0
).to(DEVICE)
test_acc = evaluate(model, test_loader, accuracy, DEVICE)
print(f"Test accuracy: {test_acc:.2%}")
evaluate_arabic_diacritization(model, test_loader, tokenizer, DEVICE)

Test accuracy: 96.33%
---  (Evaluation Results) ---
Word Error Rate (WER): 13.97%
Diacritic Error Rate (DER - شامل الأخير): 3.67%
Diacritic Error Rate (DER - بدون الحرف الأخير): 3.39%
Exact Match (Sentence Accuracy): 9.58%


{'wer': 13.97239802362793,
 'der': 3.6679264426125555,
 'der_no_last': 3.3923188436001923,
 'sentence_accuracy': 9.580638709247284}